In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1. Load environment variables from .env
load_dotenv()
URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")

AUTH = (USER, PASSWORD)

load_dotenv()
print("URI loaded:", os.getenv("NEO4J_URI") is not None)
print("User loaded:", os.getenv("NEO4J_USERNAME"))
print("Password loaded:", os.getenv("NEO4J_PASSWORD") is not None)

This cell loads Neo4j connection details from the `.env` file and prints whether each required variable is present.

Summary: Checks that `NEO4J_URI`, `NEO4J_USERNAME`, and `NEO4J_PASSWORD` are discoverable before attempting DB operations.

In [ ]:
with driver.session() as session:
    result = session.run("CALL db.labels()")
    labels = [record["label"] for record in result]
    print("Existing node labels in database:", labels)

This cell queries the database for all node labels (types) currently present in the graph.

Summary: Useful to verify that expected labels like `Preschool`, `Town`, and `CareLevel` exist before running queries.

In [ ]:
with driver.session() as session:
    result = session.run("MATCH (n) RETURN labels(n) AS label, count(n) AS count")
    for record in result:
        print(f"📊 Label: {record['label']} | Count: {record['count']}")

This cell counts nodes per label to give a quick inventory of graph contents and relative sizes.

Summary: Expect counts for `Preschool`, `Town`, and `CareLevel` if the import ran successfully; zero counts indicate missing data.

In [ ]:
with driver.session() as session:
    # Check all stored towns
    towns = session.run("MATCH (t:Town) RETURN t.name").data()
    print("🏛️ Towns in DB:", towns)

    # Check all stored care levels
    levels = session.run("MATCH (c:CareLevel) RETURN c.name").data()
    print("📚 Care Levels in DB:", levels)

This cell retrieves the distinct `Town` names and `CareLevel` names stored in the graph.

Summary: Confirms whether the expected attribute values (town codes and level names) are present for filtering in Stage 1 queries.

In [ ]:
user_town = "54"
user_level = "Pre-Nursery (3 yrs old)"

stage1_query = """
MATCH (p:Preschool)-[:LOCATED_IN]->(t:Town {name: $town})
MATCH (p)-[:SERVES_LEVEL]->(c:CareLevel {name: $level})
RETURN p.centre_code AS centre_code, p.name AS name
"""

with driver.session() as session:
    result = session.run(stage1_query, town=user_town, level=user_level)
    shortlisted_schools = result.data()

print(f"Found {len(shortlisted_schools)} matching preschools!")

This cell runs a parameterized Cypher query to find preschools located in a specific `Town` and serving a specific `CareLevel`.

Summary: The output is a list of matching preschools (centre code and name). Use these results as the Stage‑1 shortlist for Stage‑2 evaluation.

In [ ]:
# Define the parent's search preferences collected from the UI
user_town = "54"
user_level = "Nursery 1"

# Cypher query using placeholders ($town and $level) for security and flexibility
stage1_query = """
MATCH (p:Preschool)-[:LOCATED_IN]->(t:Town {name: $town})
MATCH (p)-[:SERVES_LEVEL]->(c:CareLevel {name: $level})
RETURN p.centre_code AS centre_code, p.name AS name, 1200 AS base_fee
"""
driver = GraphDatabase.driver(URI, auth=AUTH)
# Execute the query and convert results into a Python list of dictionaries 📋
with driver.session() as session:
    result = session.run(stage1_query, town=user_town, level=user_level)
    shortlisted_schools = result.data()

print(f"Found {len(shortlisted_schools)} matching preschools!")

This final cell demonstrates a variant of the Stage‑1 query that returns a `base_fee` field (static here) to illustrate how shortlist results can be enriched with cost data for Stage‑2 evaluation.

Summary: Use these results as input to Stage‑2 logic (fee/subsidy calculations). Replace the static `base_fee` with actual data columns if available.